In [2]:
import requests
import pandas as pd

headers = {'User-Agent': 'MyName is khoinguyen220707@gmail.com'}

cik = "0000320193"
url = f"https://data.sec.gov/api/xbrl/companyfacts/CIK{cik}.json"


response = requests.get(url, headers=headers)

if response.status_code == 200:
    data = response.json()
    print("Successful! The company:", data['entityName'])
else:
    print("Error:", response.status_code)

Successful! The company: Apple Inc.


In [3]:

net_income_data = data['facts']['us-gaap']['NetIncomeLoss']['units']['USD']

df = pd.DataFrame(net_income_data)

df_annual = df[df['form'] == '10-K'].copy()

df_clean = df_annual[['end', 'val']].copy()

df_clean.columns = ['Date', 'Net Income (USD)']

df_clean.tail()

,Date,Net Income (USD)
308,2023-09-30,96995000000
309,2023-09-30,96995000000
320,2024-09-28,93736000000
321,2024-09-28,93736000000
332,2025-09-27,112010000000


In [4]:
df_clean.to_csv("apple_net_income.csv", index=False)

print("File Saving completed")

File Saving completed


In [5]:
import sqlite3

conn = sqlite3.connect('financial_data.db')

print("Connected succesfully")

Connected succesfully


In [6]:
df_clean.to_sql('apple_net_income', conn, if_exists='replace', index=False)

print("SQL Upload Completed")

SQL Upload Completed


In [7]:
import pandas as pd

query = "SELECT * FROM apple_net_income ORDER BY Date DESC LIMIT 5"

df_from_sql = pd.read_sql(query, conn)
print(df_from_sql)

         Date  Net Income (USD)
0  2025-09-27      112010000000
1  2024-09-28       93736000000
2  2024-09-28       93736000000
3  2023-09-30       96995000000
4  2023-09-30       96995000000


In [ ]:
import requests
import pandas as pd
import time

headers = {'User-Agent': 'MyName is khoinguyen220707@gmail.com'}

print("Loading Ticker & CIK list from SEC...")

tickers_url = "https://www.sec.gov/files/company_tickers.json"
response_tickers = requests.get(tickers_url, headers=headers)
ticker_data = response_tickers.json()

ticker_to_cik = {}

for key, data in ticker_data.items():
    ticker_to_cik[data['ticker']] = str(data['cik_str']).zfill(10)

tickers = ['AAPL', 'MSFT', 'GOOGL', 'TSLA', 'AMZN']
all_company_data = []

for ticker in tickers:
    print(f"Loading: {ticker}...")

    cik = ticker_to_cik.get(ticker)
    
    if cik:
        url_cik = f"https://data.sec.gov/api/xbrl/companyfacts/CIK{cik}.json"
        response = requests.get(url_cik, headers=headers)
        
        if response.status_code == 200:
            data = response.json()
            try:
                us_gaap = data['facts']['us-gaap']

                if 'NetIncomeLoss' in us_gaap:
                    ni_data = us_gaap['NetIncomeLoss']['units']['USD']
                    df_ni = pd.DataFrame(ni_data)
                    df_ni['Ticker'] = ticker
                    df_ni['Metric'] = 'Net Income'
                    df_ni = df_ni[df_ni['form'] == '10-K']
                    all_company_data.append(df_ni)

                rev_key = 'Revenues' if 'Revenues' in us_gaap else ('SalesRevenueNet' if 'SalesRevenueNet' in us_gaap else None)
                if rev_key:
                    rev_data = us_gaap[rev_key]['units']['USD']
                    df_rev = pd.DataFrame(rev_data)
                    df_rev['Ticker'] = ticker
                    df_rev['Metric'] = 'Revenue'
                    df_rev = df_rev[df_rev['form'] == '10-K']
                    all_company_data.append(df_rev)

                if 'Assets' in us_gaap:
                    as_data=us_gaap['Assets']['units']['USD']
                    df_as=pd.DataFrame(as_data)
                    df_as['Ticker']= ticker
                    df_as['Metric']='Assets'
                    df_as= df_as[df_as['form']=='10-K']
                    all_company_data.append(df_as)

                if 'GrossProfit' in us_gaap:
                    gr_data=us_gaap['GrossProfit']['units']['USD']
                    df_gr=pd.DataFrame(gr_data)
                    df_gr['Ticker']=ticker
                    df_gr['Metric']='GrossProfit'
                    df_gr=df_gr[df_gr['form']=='10-K']
                    all_company_data.append(df_gr)

                if 'Liabilities' in us_gaap:
                    li_data=us_gaap['Liabilities']['units']['USD']
                    df_li=pd.DataFrame(li_data)
                    df_li['Ticker']=ticker
                    df_li['Metric']='Liabilities'
                    df_li=df_li[df_li['form']=='10-K']
                    all_company_data.append(df_li)

                if 'StockholdersEquity' in us_gaap:
                    se_data=us_gaap['StockholdersEquity']['units']['USD']
                    df_se=pd.DataFrame(se_data)
                    df_se['Ticker']=ticker
                    df_se['Metric']='StockholdersEquity'
                    df_se=df_se[df_se['form']=='10-K']
                    all_company_data.append(df_se)

                print(f"Extraction Completed for {ticker}!")
                
            except KeyError:    
                print(f"JSON error of {ticker}")

        else:
            print(f"API error {response.status_code}")
            
    time.sleep(0.2)

import pandas_gbq
from google.oauth2 import service_account

if all_company_data:
    df_final = pd.concat(all_company_data, ignore_index=True)
    df_final = df_final[['Ticker', 'end', 'Metric', 'val']]

    df_final.columns = ['Ticker', 'Date', 'Metric', 'Value_USD']
    df_final['Date'] = pd.to_datetime(df_final['Date'])

    credentials = service_account.Credentials.from_service_account_file("gcp-key.json")
    
    project_id = "first-case-study-485421" 
    table_id = "portfolio_data.sec_financial_metrics"

    print("\nPipline loading to BigQuery...")

    try:
        pandas_gbq.to_gbq(
            df_final, 
            destination_table=table_id, 
            project_id=project_id, 
            if_exists='replace',
            credentials=credentials
        )
        print("PIPELINE COMPLETED!")
        print(df_final.head())
    except Exception as e:
        print(f"Pipeline Error {e}")

Loading Ticker & CIK list from SEC...
Loading: AAPL...
Extraction Completed for AAPL!
Loading: MSFT...
Extraction Completed for MSFT!
Loading: GOOGL...
Extraction Completed for GOOGL!
Loading: TSLA...
Extraction Completed for TSLA!
Loading: AMZN...
Extraction Completed for AMZN!

 Pipline loading to BigQuery...
PIPELINE COMPLETED!
  Ticker       Date      Metric   Value_USD
0   AAPL 2007-09-29  Net Income  3496000000
1   AAPL 2008-09-27  Net Income  4834000000
2   AAPL 2008-09-27  Net Income  6119000000
3   AAPL 2008-12-27  Net Income  2255000000
4   AAPL 2009-03-28  Net Income  1620000000
